# Glaucoma v5: EfficientNet-B0 started from ODIR-pretrained weights (grouped 5-fold CV)
**What this notebook does, in order:** (1) read the public ODIR dataset and build per-eye labels, (2) remove ODIR images that are copies of project images (and **stop if any is a copy of a TEST image**), (3) pretrain EfficientNet-B0 on ODIR, (4) run the usual 5-fold x 3-seed CV at 224 px starting from those weights, (5) zip everything.

**Attach these two datasets (and nothing else):**
- `Glaucoma CV Pack (train+val)` (has the 256-px `pack/`)
- the **v5 bundle** dataset. It must contain exactly: `src/`, `prep_odir.py`, `pretrain_odir.py`, `target_hashes.csv` **and `test_thumbs_128.npz`** (128x128 thumbnails of the 369 TEST images, used only to tell real duplicates from hash collisions, never for training or evaluation)
- plus the public ODIR dataset `andrewmvd/ocular-disease-recognition-odir5k` (Add Input -> search the name)

**PRETRAIN_SEED=7 is the replication run** (a second ODIR pretraining with a different seed; the CV settings are identical to the seed-42 run, and its results go to their own `..._s7` folder and zip).

Kaggle settings: **GPU T4, Internet On** (ImageNet weights are downloaded once). Only the starting weights differ from the ImageNet control run; folds, seeds and recipe are identical.

**De-duplication rule (same as Notebook 1: hash first, then pixels):** an ODIR image is a candidate when its dHash is within 3 bits of a project image. Candidates are then compared by pixels (MAE on 128x128 images).
- vs train/val: MAE < 5.0 -> duplicate, dropped.
- vs TEST: MD5 identical or MAE <= 5.0 -> **confirmed test duplicate: dropped and the notebook STOPS**. MAE > 5.0 -> hash collision, dropped to be safe, **no stop** (marked `drop_TEST_candidate_hash_collision` in `odir_dedupe_audit.csv`).
Read the Hamming distance and MAE of every candidate in the printout and in `odir_dedupe_audit.csv`.

In [ ]:
# ---- Settings you may change ----
INIT = "binary"        # "binary" = ODIR Glaucoma-vs-Normal pretraining (primary) | "multi" = 8-category multi-label (secondary)
PRETRAIN_SEED = 7      # seed of the ODIR pretraining (split, weights init of the head, shuffling). 42 = first run, 7 = replication run
QUICK_TEST = True      # True = tiny ODIR subset, 1 epoch, 2 folds, 1 seed (a few minutes). Set False for the real run.
USE_AMP = True         # mixed precision on the GPU
SAVE_CHECKPOINTS = False
assert INIT in ("binary", "multi")

In [ ]:
# ---- Find code, project pack, target hashes and the ODIR dataset ----
import sys, glob, os
from pathlib import Path
if os.path.isdir('/kaggle/input'):
    hits = glob.glob('/kaggle/input/**/src/odir.py', recursive=True)
    assert hits, 'v5 code not found: is the v5 bundle dataset attached?'
    CODE_DIR = Path(hits[0]).parents[1]
    csvs = [p for p in glob.glob('/kaggle/input/**/full_df.csv', recursive=True) + glob.glob('/kaggle/input/**/data.csv', recursive=True)]
    csvs = sorted(csvs, key=lambda p: ('odir' not in p.lower(), 'full_df' not in p))
    assert csvs, 'ODIR csv (full_df.csv / data.csv) not found: is the ODIR dataset attached?'
    ODIR_ROOT = Path(csvs[0]).parent
    WORK = Path('/kaggle/working')
else:   # local smoke test: point ODIR_ROOT at a fake ODIR folder
    CODE_DIR, ODIR_ROOT, WORK = Path.cwd(), Path(os.environ['ODIR_ROOT']), Path.cwd() / 'v5_work'
sys.path.insert(0, str(CODE_DIR))
import torch
from src.config import find_pack_dir
PACK = find_pack_dir('pack')                  # the 256-px project pack
TARGET_HASHES = CODE_DIR / 'target_hashes.csv'
if not os.path.isdir('/kaggle/input'):         # local smoke tests may swap in fake hashes / pack
    PACK = Path(os.environ.get('PACK_DIR', PACK))
    TARGET_HASHES = Path(os.environ.get('TARGET_HASHES', TARGET_HASHES))
assert TARGET_HASHES.exists(), 'target_hashes.csv missing from the v5 bundle'
assert TARGET_HASHES.with_name('test_thumbs_128.npz').exists(), 'test_thumbs_128.npz missing from the v5 bundle (must sit next to target_hashes.csv)'
print('code:', CODE_DIR, '\nODIR:', ODIR_ROOT, '\npack:', PACK, '\nCUDA:', torch.cuda.is_available())

In [ ]:
import zipfile
from src.odir import prepare_odir
from src.pretrain import pretrain
from src.config import Config
from src.cv import run_cv

ODIR_OUT, ODIR_IMG = WORK / 'odir_out', WORK / 'odir_pack'
TAG = {'binary': 'odirbin', 'multi': 'odirmulti'}[INIT]
SEED_TAG = '' if PRETRAIN_SEED == 42 else f'_s{PRETRAIN_SEED}'
cfg = None
try:
    # 1-2. ODIR labels, preprocessing, de-duplication against the project images
    prep = prepare_odir(ODIR_ROOT, TARGET_HASHES, PACK, ODIR_OUT, ODIR_IMG, quick_n=60 if QUICK_TEST else None, arm=INIT)   # only this arm's eyes are preprocessed
    if prep['test_matches']:
        print('\n' + '#' * 72 + '\n#  STOPPING: an ODIR image matches a project TEST image. Audit CSVs are written.\n' + '#' * 72)
        raise RuntimeError('ODIR/TEST overlap found - see odir_dedupe_audit.csv. Not continuing.')
    # 3. pretraining (backbone weights + ODIR validation metrics)
    pt, odir_metrics = pretrain(INIT, ODIR_OUT / 'odir_manifest.csv', ODIR_IMG / 'images', ODIR_OUT,
                                quick=QUICK_TEST, amp=USE_AMP, seed=PRETRAIN_SEED)
    # 4. the usual grouped CV, only the starting weights differ from the ImageNet control
    cfg = Config(arch='efficientnet_b0', res=224, quick_test=QUICK_TEST, use_amp=USE_AMP,
                 save_checkpoints=SAVE_CHECKPOINTS, init_weights=pt, init_tag=TAG, pretrain_seed=PRETRAIN_SEED)
    print('PRETRAIN_SEED:', PRETRAIN_SEED, '| backbone:', pt.name, '| CV output folder:', cfg.out_dir)
    all_ok = run_cv(cfg)
    print('ALL CHECKS PASSED' if all_ok else 'SOME CHECKS FAILED - read the log above')
finally:
    # 5. zip whatever exists (CV outputs, ODIR audit CSVs, backbone .pt, ODIR metrics)
    zpath = WORK / f"v5_{INIT}{SEED_TAG}{'_quick' if QUICK_TEST else ''}_outputs.zip"
    with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as z:
        for folder in [ODIR_OUT] + ([cfg.out_dir] if cfg else []):
            if Path(folder).exists():
                for f in Path(folder).rglob('*'):
                    if f.is_file():
                        z.write(f, Path(folder).name + '/' + str(f.relative_to(folder)))
    print('\n' + '=' * 70 + '\nZip created:', zpath, f'({zpath.stat().st_size/1e6:.1f} MB)')
    print('DOWNLOAD THE ZIP NOW; the session wipes /kaggle/working when it stops')
    print('=' * 70)

In [ ]:
# ---- Results ----
import json
from IPython.display import Image, display
print(json.dumps(json.load(open(ODIR_OUT / (f'odir_{INIT}_metrics' + SEED_TAG + '.json'))), indent=2))
for f in ['fig_pooled_roc_pr.png', 'fig_per_fold_auc.png']:
    display(Image(filename=str(cfg.out_dir / f)))
res = json.load(open(cfg.out_dir / 'results.json'))
print(json.dumps({k: res[k] for k in ['init', 'pretrain_seed', 'arch', 'img_size', 'mean_std_across_folds', 'pooled_out_of_fold']}, indent=2))
print('\nREMINDER: DOWNLOAD THE ZIP NOW (Output tab); the session wipes /kaggle/working when it stops')